# R06 · Cookbook Reference Lab: Configuration, maintenance, files, errors, system space, CLI

**NHyDB 0.28.2 notebook series · v1 · reference part**

These examples are based on the official
[NHyDB Python Cookbook](https://github.com/nhydb-run4/NHyDB/blob/main/docs/cookbook/PYTHON_COOKBOOK.md) (sections 19–24), which serves as their authoritative reference.
Code and expected output are taken unchanged from the cookbook, and the same examples are run by the
NHyDB test suite.

* Every code cell is an **independent example** – run any cell on its own, or *Run All*.
* Each example creates its own temporary database folder.
* Under each example you find the output printed in the cookbook, to compare with yours.
* The main series (`../main/`) explains the ideas behind these APIs step by step.

**Contents** (11 runnable examples)

* 19. Configuration and limits
* 20. Maintenance: stats, health, verify, repair, compaction
* 21. Database files, backup, restore and moving
* 22. Errors, locks and concurrent access
* 23. System space R0,u1
* 24. Command line

In [1]:
import nhydb

print('nhydb', nhydb.ENGINE_VERSION, '· file format', nhydb.FILE_FORMAT_VERSION)
if nhydb.ENGINE_VERSION != '0.28.2':
    print('⚠ These examples were written for NHyDB 0.28.2; results may differ.')

nhydb 0.28.2 · file format 0.28.2


## 19. Configuration and limits

Settings can come from a TOML file. The engine looks in this order:
`config=` argument, environment variable `NHYDB_CONFIG`,
`nhydb_config.toml` next to the database, built-in defaults. A commented
template is `config/nhydb_config.example.toml` in the repository.

In [2]:
import os
import tempfile

import nhydb

folder = tempfile.mkdtemp()
with open(os.path.join(folder, "nhydb_config.toml"), "w", encoding="utf-8") as f:
    f.write('[database]\ndb_max_size_bytes = 300000\ndurability = "full"\n\n[lock]\ntimeout_ms = 2000\n')

path = os.path.join(folder, "configured.nhydb")
with nhydb.create(path) as db:                       # the file next to the database is found
    print(os.path.basename(db.config_source), db.durability, db.max_db_size_bytes)
    db.create_realm(1, "r")
    try:
        db.put_points(1, [(u, 1, b"x" * 1000, "bytes") for u in range(1, 400)])
    except nhydb.DatabaseSizeLimitError:
        print("size limit reached")

nhydb_config.toml full 300000
size limit reached


*Expected output (from the cookbook):*

```text
nhydb_config.toml full 300000
size limit reached
```

| Section / key | Meaning |
|---|---|
| `[database] db_max_size_bytes` | size limit for **new** databases (default 50 GiB, at least 64 KiB) |
| `[database] durability` | `"fsync"` or `"full"` (aliases `"default"`/`"normal"`, `"fullfsync"`) |
| `[math] realm_auto_limit` | realm budget for new MATH layouts (default 4096) |
| `[math] math_max_intermediate_bits` | bit limit of MATH expressions, frozen into each new MATH layout |
| `[lock] enabled`, `timeout_ms` | lock file and wait time |
| `[paths] layout_path`, `key_path` | path templates, `{db}` = database file name, `{db_path_hash}` = hash of the absolute database path (for equally named databases sharing a key folder) |
| `[cache] block_cache_bytes` | budget of the block cache per open object (default 64 MiB, LRU; `0` = no cache); blocks up to max(8 MiB, budget/8) are cached |
| `[cache] vindex_max_marks`, `vindex_max_indexes` | budget of the cached v-indexes of large sparse u groups (in jump marks) and their number |

Explicit arguments win over the file (`lock=`/`lock_timeout_ms=` override
the `[lock]` keys one by one). Values stored in the database (the size
limit, the realm budget and bit limit of a MATH layout) are never
overridden when opening. Unknown keys are errors. Change the size limit of an existing
database with `set_max_db_size`:

In [3]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "limit.nhydb")
with nhydb.create(path) as db:
    db.set_max_db_size(10 * 1024**2, commit=True)     # 10 MiB
with nhydb.open(path) as db:
    print(db.max_db_size_bytes)

10485760


*Expected output (from the cookbook):*

```text
10485760
```

**Block cache.** Every block is read completely and CRC-checked once per
open object; then it is either kept in the block cache or, if it does not
fit, later read only in the parts that are needed. `stats()["block_cache"]`
shows the budget, the largest cacheable block, the cached and verified
blocks and the counters `hits`, `loads`, `partial_reads` and `evictions`:

In [4]:
import os
import tempfile

import nhydb

folder = tempfile.mkdtemp()
with open(os.path.join(folder, "nhydb_config.toml"), "w", encoding="utf-8") as f:
    f.write("[cache]\nblock_cache_bytes = 16777216\n")      # 16 MiB block cache

path = os.path.join(folder, "cached.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "x", "type": "u32"}])
    db.create_classic_layout(1, 1, "c", shape_id=1)
    db.put_dense_batch(1, 1, [[u] for u in range(1, 10_001)], commit=True)

with nhydb.open(path, readonly=True) as db:
    for u in (1, 5_000, 10_000):
        db.get(1, u)
    bc = db.stats()["block_cache"]
    print(bc["budget_bytes"], bc["max_item_bytes"])
    print(bc["loads"], bc["hits"], bc["cached_blocks"], bc["verified_blocks"])

16777216 8388608
1 2 1 1


*Expected output (from the cookbook):*

```text
16777216 8388608
1 2 1 1
```

## 20. Maintenance: stats, health, verify, repair, compaction

In [5]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "maint.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "x", "type": "u32"}])
    db.create_classic_layout(1, 1, "c", shape_id=1)
    for u in range(1, 201):
        db.put(1, u, [u], commit=True)                # many small commits -> many blocks

    st = db.stats()
    print(st["format"], st["storage"]["physical_dense_records"], st["file_bytes"] > 0)
    h = db.health(deep=True)
    print(h["status"], [(c["name"], c["status"]) for c in h["checks"] if c["status"] != "ok"])
    offset = db.checkpoint()                          # writes a CHECKPOINT block and commits
    print(offset == db.checkpoint_offset, db.health()["status"])
    v = db.verify(deep=True)
    print(v["ok"], v["error_count"])

print(nhydb.NHyDB.inspect_tail(path)["clean"])
res = nhydb.NHyDB.compact_file(path, make_backup=False)   # rewrite the live state compactly
print(res["ok"], res["records_written"], res["after_bytes"] < res["before_bytes"])
with nhydb.open(path) as db:
    print(db.get(1, 200)["x"])

0.28.2 200 True
warning [('checkpoint', 'warning')]
True healthy
True 0
True
True 200 True
200


*Expected output (from the cookbook):*

```text
0.28.2 200 True
warning [('checkpoint', 'warning')]
True healthy
True 0
True
True 200 True
200
```

* `verify(deep=False)` checks the block structure (`deep=True` also the
  payload checksums), continues after damage and lists every damaged
  block; stale indexes appear under `warnings` (they are not errors).
* `health(deep=False)` runs the checks `container`, `layouts`, `spacemap`,
  `layout_routing`, `math_layouts`, `definitions`, `indexes` and
  `checkpoint`; status `healthy`, `warning` or `unhealthy` (a stale index
  is a `warning`). A database without a recent checkpoint reports a
  `checkpoint` warning;
  `checkpoint()` writes one (a diagnostic snapshot of the segment
  structure; it does not replace older blocks).
* `stats()` reports file, format, realms, shapes, layouts, extensions,
  indexes and storage statistics, and under `block_cache` the block cache
  of this object (section 19).
* `compact_file(path, output=None, make_backup=True, ...)` is static and
  needs the database to be closed. In place it keeps the identity and side
  files and writes a backup `<db>.before_compact_<timestamp>` plus
  `<db>.before_compact_<timestamp>.layout.enc` (unless
  `make_backup=False`). With `output=` it exports to a **new** database
  with its own side files. A failed compaction leaves the source unchanged
  and removes only the files it created. Memory use is bounded by one
  batch (`batch_records`), not by the size of the database: records are
  copied as raw bytes and sparse cells in chunks.

**Repairing a damaged tail.** `inspect_tail` checks the end of the file
without opening it; `repair_file(path, apply=False)` only reports,
`apply=True` cuts off the damaged tail (with a backup `<db>.bak_<timestamp>`
by default):

In [6]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "damaged.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.put_cell(1, 1, 1, "kept", type="utf8", commit=True)

with open(path, "ab") as f:
    f.write(b"NHBL" + b"\x07" * 40)                  # simulate an incomplete write

tail = nhydb.NHyDB.inspect_tail(path)
print(tail["clean"], tail["trailing_bad_bytes"])
print(nhydb.NHyDB.repair_file(path)["clean"])        # report only
rep = nhydb.NHyDB.repair_file(path, apply=True, make_backup=False)
print(rep["applied"], nhydb.NHyDB.inspect_tail(path)["clean"])
with nhydb.open(path) as db:
    print(db.get_value(1, 1, 1))

False 44
False
True True
kept


*Expected output (from the cookbook):*

```text
False 44
False
True True
kept
```

`repair_file` never sacrifices committed data silently:

* If intact committed blocks lie behind a damaged spot, it raises
  `CorruptionError` and changes nothing. `allow_data_loss=True` cuts them
  anyway and lists them in `lost_blocks`.
* Uncommitted blocks after the commit boundary are discarded
  (`discarded_uncommitted_blocks`); `keep_uncommitted=True` keeps intact
  ones (`kept_uncommitted_blocks`).
* If one superblock is damaged and complete blocks follow the boundary, you
  must decide explicitly with one of these two switches.
* The dry run (`apply=False`) reports the planned `action` (`none`,
  `repair` or `abort`) and, if it would refuse, `plan_error`.
* The backup consists of `<db>.bak_<timestamp>` and
  `<db>.bak_<timestamp>.layout.enc`.

Opening a database with a damaged tail also works – the writer cuts off
everything after the commit boundary automatically, as long as both
superblocks agree. If they do not (only possible after a power or system
failure while a superblock was written – a killed process never causes
it), opening fails without changing the file: `nhydb.open(path,
readonly=True)` still reads the previous committed state, and
`repair_file(path, apply=True, keep_uncommitted=True)` keeps the intact
blocks after the boundary (`allow_data_loss=True` discards them).

To go back to a backup of `repair_file` or `compact_file`, call
`nhydb.NHyDB.restore_backup(backup_path, path)` (command line:
`nhydb restore BACKUP DB`). It checks that the backup belongs to the
database and opens with its key, then replaces the data file and the layout
plan; the key file stays as it is.

## 21. Database files, backup, restore and moving

| File | Content | Note |
|---|---|---|
| `X.nhydb` | data (append-only container) | not encrypted |
| `X.nhydb.layout.enc` | schema (realms, shapes, layouts, indexes, MATH), encrypted | |
| `X.nhydb.layout.key` | 32-byte key for the schema file | **without it the database cannot be opened**; Unix permissions `0600` |
| `X.nhydb.lock` | lock file, empty | may be deleted while the database is closed |

Rules:

* **Back up all three files together** (`.nhydb`, `.layout.enc`,
  `.layout.key`) while the database is **closed**. Keep a copy of the key
  file in a safe place; a lost key cannot be recovered.
* The file name is part of the database identity: a renamed `.nhydb` does
  not open (also not a change of upper/lower case only). Restore backups
  under the **same file name** (another folder is fine). To create a copy
  under a new name use `compact_file(path, output=new_path)`.
* Only the schema is encrypted. The data in `X.nhydb` is stored in plain
  form; protect the files with file-system permissions or disk encryption.
* The key file is created with permissions `0600` on Unix; if it is
  readable by group or others, opening issues a `RuntimeWarning` (also in
  `db.open_warnings`). **Windows:** the key file inherits the ACL of its
  folder – NHyDB sets no ACL of its own. Keep key files in a folder only you
  can access (for example below `%USERPROFILE%`) or point `key_path` there.
* A backup copy on read-only media can be inspected with
  `nhydb.open(path, readonly=True)`.
* **Databases of NHyDB 0.28.0, 0.27.1 or earlier do not open** with 0.28.2, also
  not with `readonly=True`: opening fails closed with `CorruptionError`
  (the message names the format) and changes nothing. There is no migration and
  `repair_file` cannot convert such a file (its dry run reports
  `action: "abort"`). Export the data with the release that wrote it and re-import it
  into a new database with 0.28.2.

In [7]:
import os
import shutil
import tempfile

import nhydb

live = tempfile.mkdtemp()
path = os.path.join(live, "crm.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "crm")
    db.put_cell(1, 1, 1, "customer 1", type="utf8")

# backup: copy the three files of the closed database
backup = tempfile.mkdtemp()
for suffix in ("", ".layout.enc", ".layout.key"):
    shutil.copy2(path + suffix, os.path.join(backup, "crm.nhydb" + suffix))

# restore into another folder, same file name
restored = tempfile.mkdtemp()
for suffix in ("", ".layout.enc", ".layout.key"):
    shutil.copy2(os.path.join(backup, "crm.nhydb" + suffix), os.path.join(restored, "crm.nhydb" + suffix))
with nhydb.open(os.path.join(restored, "crm.nhydb")) as db:
    print(db.get_value(1, 1, 1))

# a renamed file is rejected ...
os.rename(os.path.join(restored, "crm.nhydb"), os.path.join(restored, "renamed.nhydb"))
try:
    nhydb.open(os.path.join(restored, "renamed.nhydb"),
               layout_path=os.path.join(restored, "crm.nhydb.layout.enc"),
               key_path=os.path.join(restored, "crm.nhydb.layout.key"))
except nhydb.CorruptionError:
    print("renamed file rejected")

# ... use compact_file(output=...) for a copy under a new name
copy = os.path.join(tempfile.mkdtemp(), "crm_copy.nhydb")
nhydb.NHyDB.compact_file(path, output=copy)
print(sorted(os.listdir(os.path.dirname(copy))))
with nhydb.open(copy) as db:
    print(db.get_value(1, 1, 1))

customer 1
renamed file rejected
['crm_copy.nhydb', 'crm_copy.nhydb.layout.enc', 'crm_copy.nhydb.layout.key', 'crm_copy.nhydb.lock']
customer 1


*Expected output (from the cookbook):*

```text
customer 1
renamed file rejected
['crm_copy.nhydb', 'crm_copy.nhydb.layout.enc', 'crm_copy.nhydb.layout.key', 'crm_copy.nhydb.lock']
customer 1
```

**Keep the key file somewhere else.** Pass `key_path` (and optionally
`layout_path`) when creating and every time you open the database – or put
the path templates into the configuration file (section 19):

In [8]:
import os
import tempfile

import nhydb

data_dir, key_dir = tempfile.mkdtemp(), tempfile.mkdtemp()
path = os.path.join(data_dir, "secure.nhydb")
key = os.path.join(key_dir, "secure.nhydb.layout.key")

with nhydb.create(path, key_path=key) as db:
    db.create_realm(1, "r")
print(sorted(os.listdir(data_dir)), os.listdir(key_dir))

try:
    nhydb.open(path)                                    # key not found next to the database
except OSError:
    print("key missing -> cannot open")
with nhydb.open(path, key_path=key) as db:
    print(1 in db.realms)

['secure.nhydb', 'secure.nhydb.layout.enc', 'secure.nhydb.lock'] ['secure.nhydb.layout.key']
key missing -> cannot open
True


*Expected output (from the cookbook):*

```text
['limiter.txt', 'secure.nhydb', 'secure.nhydb.layout.enc', 'secure.nhydb.lock'] ['secure.nhydb.layout.key']
key missing -> cannot open
True
```

## 22. Errors, locks and concurrent access

All engine errors derive from `nhydb.NHyDBError`, including the MATH
expression errors (`MathError`), the MATH realm planner (`LayoutError`)
and the query layer (`QueryError`). Only `IndexConfigError` is a
`ValueError` instead.

```text
NHyDBError
├── CorruptionError, SchemaError, StaleIndexError, RealmLimitError, AddressError
├── DatabaseLockedError, DatabaseSizeLimitError
├── ResourceLimitError (reason: cancelled_by_app, cancelled_by_limiter, wait_timeout)
├── MathLayoutError ── MathLayoutNotFound, MappingFrozenError, MathCellConflictError
├── MathError ── MathSyntaxError, MathDomainError
├── LayoutError ── LayoutStateError, RealmBudgetExceeded, StalePlanError, AllocationConflictError
└── QueryError ── QueryPlanError, QueryDomainError, SQLSyntaxError
ValueError ── IndexConfigError
```

The engine also raises built-in exceptions:

* `ValueError`, `TypeError`, `KeyError` for invalid arguments (for example
  `TypeError` for `u=2.7`, `ValueError` for IDs above 2^63 − 1 or values
  nested deeper than 256 levels);
* `OverflowError` when a Python integer does not fit a 64-bit parameter
  (for example `u_window=2**64`), `UnicodeEncodeError` for text with lone
  surrogates;
* `MemoryError` when a result cannot fit into memory;
* `OSError` subclasses with `errno` and, where known, `filename`:
  `FileNotFoundError`, `FileExistsError`, `PermissionError`,
  `IsADirectoryError` …;
* `RuntimeError` when one object is used by two threads at the same time,
  or when a plugin calls back into a database that is writing.

Damaged files always raise `CorruptionError`.

In [9]:
import nhydb

print(issubclass(nhydb.CorruptionError, nhydb.NHyDBError))
print(issubclass(nhydb.MathSyntaxError, nhydb.NHyDBError), issubclass(nhydb.MathSyntaxError, nhydb.MathError))
print(issubclass(nhydb.SQLSyntaxError, nhydb.QueryError), issubclass(nhydb.IndexConfigError, ValueError))
try:
    nhydb.n_to_dynadron_address(576)
except nhydb.NHyDBError as e:
    print(type(e).__name__)

True
True True
True True
AddressError


*Expected output (from the cookbook):*

```text
True
True True
True True
AddressError
```

**One writing object per file.** Opening a database for writing takes an
exclusive lock on `<db>.lock`. A second such open (in the same or in
another process) raises `DatabaseLockedError`, unless it waits with
`lock_timeout_ms`. Readers that must not block can open with
`readonly=True`: no lock, nothing in the file is changed, write calls and
`commit()` raise `NHyDBError`, and the object sees the committed state at
the time it was opened (later commits of the writer are not reloaded).

In [10]:
import os
import subprocess
import sys
import tempfile
import textwrap

import nhydb

path = os.path.join(tempfile.mkdtemp(), "shared.nhydb")
nhydb.create(path).close()

with nhydb.open(path) as db:
    try:
        nhydb.open(path)                                # same process, second object
    except nhydb.DatabaseLockedError:
        print("locked (same process)")
    with nhydb.open(path, readonly=True) as reader:     # readers need no lock
        print("read-only:", reader.readonly)

holder = textwrap.dedent("""
    import sys, time, nhydb
    db = nhydb.open(sys.argv[1])
    print("locked", flush=True)
    time.sleep(1.0)
    db.close()
""")
proc = subprocess.Popen([sys.executable, "-c", holder, path], stdout=subprocess.PIPE, text=True)
assert proc.stdout.readline().strip() == "locked"
try:
    nhydb.open(path)
except nhydb.DatabaseLockedError:
    print("locked (other process)")
db = nhydb.open(path, lock_timeout_ms=10_000)           # waits until the other process closes
print("opened after waiting")
db.close()
proc.wait(timeout=30)
proc.stdout.close()

locked (same process)
read-only: True
locked (other process)


opened after waiting


*Expected output (from the cookbook):*

```text
locked (same process)
read-only: True
locked (other process)
opened after waiting
```

`lock=False` disables the lock file. Only use it when you can guarantee
that no other process opens the database at the same time; with
`lock=False` an uncommitted tail is never cut off (`DatabaseLockedError`).
The lock file never follows a symbolic link.

**Threads.** Long calls release the GIL while the Rust core works: queries
(`query`, `sql`, `sql_numpy`, `explain_query`, `QueryEngine` methods),
`stats`, `health`, `verify`, `read_numpy`, the batch writers, `commit`,
`rollback` and the static file operations. Python callbacks (query
providers, layout plugins) take the GIL back for their call. Short calls
such as `get` and `find_u` keep the GIL on purpose (handing it over would
cost more than the call). Several threads may run read-only queries on the
same object at the same time; a write, `commit` or `rollback` while another
call holds the object fails immediately with `RuntimeError`
("Already borrowed") – it does not wait and changes nothing. For parallel
work give each thread its own object (readers with `readonly=True`; four
threads with own objects run SQL scans about 2.9 times faster than one) or
protect a shared object with a `threading.Lock`.

## 23. System space R0,u1

Every database stores its identity in the reserved cells `(0, 1, v)`:
database id, file names, schema generation and hash, creation time,
encryption scheme and the fixed constants.

In [11]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "sys.nhydb")
with nhydb.create(path) as db:
    meta = db.get_system_metadata()
    print(meta["realm"], meta["u"], sorted(meta["cells"], key=int))
    print(db.get_system_cell(2), db.get_system_cell(8))
    print(db.get_system_cell(9)["realm_span"] == 10**100)

0 1 ['1', '2', '3', '4', '5', '6', '7', '8', '9']
sys.nhydb AES-256-GCM + HMAC-SHA256
True


*Expected output (from the cookbook):*

```text
0 1 ['1', '2', '3', '4', '5', '6', '7', '8', '9']
sys.nhydb AES-256-GCM + HMAC-SHA256
True
```

## 24. Command line

The package installs the command `nhydb` (also `python -m nhydb`) and the
query tool `python -m nhydb.query`. `--help` shows all options; `--json`
gives machine-readable output.

```bash
nhydb to-n --realm 1 --u 11 --v 13 --lane NR
nhydb from-n N589
nhydb get shop.nhydb u2,v3 --realm 1 --mode cell
nhydb stats shop.nhydb
nhydb health shop.nhydb --deep
nhydb verify shop.nhydb
python -m nhydb.query shop.nhydb --sql "SELECT COUNT(*) AS n FROM orders" --json
python -m nhydb.query shop.nhydb --realm 1 --eq "status=paid" --return count
```

Commands: `to-n`, `from-n`, `layout`, `layout-check`, `layout-plan`,
`evolve-layout`, `route`, `describe`, `logical-address`, `occupied`,
`stats`, `health`, `get`, `address`, `cell`, `cell-n`, `find`, `verify`,
`rebuild-indexes`, `repair`, `restore`, `compact`. The CLI uses the default
side-file locations or the configuration file. Read commands open the
database with `readonly=True` and never change the file; `verify` also works
on damaged files. `repair` is a dry run unless `--apply` is given
(`--allow-data-loss`, `--keep-uncommitted` as in `repair_file`). Exit codes:
`0` success, `1` a check found problems, `2` usage error or refused
operation, `3` error (message on stderr, no traceback). In JSON output
`bytes` appear as `{"$bytes_b64": "…"}`. The same calls from Python:

In [12]:
import json
import os
import subprocess
import sys
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "cli.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "shop")
    db.create_shape(1, 1, "order", [{"v": 1, "name": "id", "type": "varuint"},
                                    {"v": 2, "name": "status", "type": "utf8"}])
    db.create_classic_layout(1, 1, "orders", shape_id=1)
    db.put_dense_batch(1, 1, [[1, "paid"], [2, "new"], [3, "paid"]])


def run(*args):
    return subprocess.run([sys.executable, *args], capture_output=True, text=True, check=True).stdout


print(json.loads(run("-m", "nhydb", "to-n", "--realm", "1", "--u", "11", "--v", "13", "--lane", "NR", "--json"))["N"])
print(json.loads(run("-m", "nhydb", "get", path, "u2,v2", "--realm", "1", "--mode", "cell", "--json"))["value"])
out = json.loads(run("-m", "nhydb.query", path, "--sql", "SELECT COUNT(*) AS n FROM orders WHERE status = ?",
                     "--params", '["paid"]', "--json"))
print(out["rows"])

589
new
[{'n': 2}]


*Expected output (from the cookbook):*

```text
589
new
[{'n': 2}]
```